<div style="display: flex; align-items: center; padding: 20px; background-color: #f0f2f6; border-radius: 10px; border: 2px solid #007bff;">
    <img src="../logo.png" style="width: 80px; height: auto; margin-right: 20px;">
    <div style="flex: 1; text-align: left;">
    <h1 style="color: #007bff; margin-bottom: 5px;">GLY 6739.017S26: Computational Seismology</h1>
    <h3 style="color: #666;">Notebook 110: Seisan to SDS converter</h3>
    <p style="color: red;"><i>Glenn Thompson | Spring 2026</i></p>
    </div>
</div>

What this notebook does:
- Reads the MVO Seisan archive, which consists of 20-minute continuous data files for all SEED ids in the network (e.g. 19 SEED ids in one each file, but 72 files per day)
- Writes to an SDS archive, which consists of day-long MiniSEED files for each SEED id (e.g. 19 files per day)
- Seisan archive is on the server
- SDS is written on your computer = faster to access

In [ ]:
import sys
from pathlib import Path

from obspy import UTCDateTime
from flovopy.research.mvo.archive import MVOSeisanArchive
from flovopy.enhanced.sdsclient import EnhancedSDSClient

# -----------------------------------------------------------------------------
# Local path setup
# -----------------------------------------------------------------------------
sys.path.append("../week8")  # this is where set_samba_data_root.py lives
from set_samba_data_root import DATA_ROOT  # noqa: E402

MVO_ROOT = Path(DATA_ROOT) / "SEISAN_DB"   # root of the MVO Seisan archive
mvo = MVOSeisanArchive(MVO_ROOT)

sds_root = Path("~/work/SDS").expanduser()
sds_root.mkdir(parents=True, exist_ok=True)
client = EnhancedSDSClient(str(sds_root))

# -----------------------------------------------------------------------------
# Time range to convert: one UTC day at a time
# -----------------------------------------------------------------------------
start_day = UTCDateTime(2003, 7, 1, 0, 0, 0)   # inclusive
end_day   = UTCDateTime(2003, 7, 14, 0, 0, 0)   # exclusive

# -----------------------------------------------------------------------------
# Conversion loop
# -----------------------------------------------------------------------------
day_start = start_day
all_written = []

while day_start < end_day:
    day_end = day_start + 86400  # 1 day

    print("=" * 80)
    print(f"Reading day: {day_start.date}  ({day_start} to {day_end})")

    st = mvo.read_continuous_stream(
        day_start,
        day_end,
        verbose=False,
        seismic_only=True,
        vertical_only=True,
        merge=True,          # use smart_merge during read
    )

    print(st)

    if len(st) == 0:
        print("No data found for this day.")
        day_start = day_end
        continue

    written = client.write_stream(
        st,
        #mode="merge",        # merge with any SDS files already written
        preprocess=False,    # stream already merged/read the way you want
        verbose=True,
        reclen=4096,
        # do not force encoding; let ObsPy choose
    )

    print(f"Wrote {len(written)} SDS files for {day_start.date}")
    for p in written[:10]:
        print("  ", p)

    all_written.extend(written)
    day_start = day_end

print("=" * 80)
print(f"Total SDS files written: {len(all_written)}")
print(f"SDS root: {sds_root}")


